# Proteogenomic Dataset Merging: pRNA Matrix Integration

This notebook implements a formal protocol for merging two large-scale heterogeneous proteogenomic dataset collections into a unified matrix. 

## Context & Data Sources

### Collection A: "Paper 1" (Bulk Protein-mRNA)
- **Source:** [Evaluation of machine learning models on protein level inference](https://www.ebi.ac.uk/biostudies/studies/S-BSST733)
- **Composition:** ~2,500 samples, 20 datasets (CPTAC, CNHPP, BrainSpan)
- **Platforms:** TMT, iTRAQ, Label-free

### Collection B: "Paper 2" (scTranslator/Generative Model)
- **Source:** A pre-trained large generative model for proteogenomics
- **Composition:** 72 datasets, 31 cancer types, 18,227 samples

#### Key Cohorts (CPTAC & TCGA):
- [Breast (CPTAC 2020)](https://www.cbioportal.org/study/summary?id=brca_cptac_2020)
- [Endometrial (CPTAC 2020)](https://www.cbioportal.org/study/summary?id=ucec_cptac_2020)
- [PDAC (CPTAC 2021)](https://www.cbioportal.org/study/summary?id=paad_cptac_2021)
- [LUAD (CPTAC 2020)](https://www.cbioportal.org/study/summary?id=luad_cptac_2020)
- [LUSC (CPTAC 2021)](https://www.cbioportal.org/study/summary?id=lusc_cptac_2021)
- [GBM (CPTAC 2021)](https://www.cbioportal.org/study/summary?id=gbm_cptac_2021)
- [Pediatric Brain](https://www.cbioportal.org/study/summary?id=brain_cptac_2020)
- [Bladder (MSK/TCGA)](https://www.cbioportal.org/study/summary?id=blca_msk_tcga_2020)
- [CCLE (Broad 2019)](https://www.cbioportal.org/study/summary?id=ccle_broad_2019)
- [TCGA Datasets](https://www.cbioportal.org/datasets)

## 1. Data Ingestion & Structure

We initialize our data ingestion by mapping the local file paths to the source collections. The input data is assumed to be in pandas DataFrames with **Genes as rows** and **Samples as columns**.

In [ ]:
import pandas as pd
import numpy as np

# 1. Mapping local file paths to datasets
dataset_paths = {
    "Collection_A": "path/to/collection_a_data.csv",
    "Collection_B": "path/to/collection_b_data.csv"
}

def load_datasets(paths):
    """
    Loads datasets from paths. Placeholder for actual CSV loading.
    Returns dummy dataframes for demonstration.
    """
    # Example Data: Genes (rows) x Samples (cols)
    genes = [f"Gene_{i}" for i in range(100)]
    samples_a = [f"Sample_{i}" for i in range(10)] + ["Overlap_1", "Overlap_2"]
    samples_b = [f"Sample_{i}" for i in range(20, 30)] + ["Overlap_1", "Overlap_2"]
    
    # Loading logic would go here:
    # df_a = pd.read_csv(paths['Collection_A'], index_col=0)
    # df_b = pd.read_csv(paths['Collection_B'], index_col=0)
    
    df_a = pd.DataFrame(np.random.randn(100, 12), index=genes, columns=samples_a)
    df_b = pd.DataFrame(np.random.randn(100, 12), index=genes, columns=samples_b)
    
    return df_a, df_b

df_a, df_b = load_datasets(dataset_paths)
print(f"Collection A Shape: {df_a.shape}")
print(f"Collection B Shape: {df_b.shape}")

## 2. Overlap Resolution

Both collections contain CPTAC data. We implement a check for duplicate samples based on Sample ID string matching. 

**Rule:** If a sample exists in both collections, we prioritize the version from **Collection B** (Paper 2) as it is part of the larger, more recent curation.

In [ ]:
def resolve_overlaps(df_a, df_b):
    """
    Identifies duplicate samples and prioritizes Collection B.
    """
    overlap_samples = set(df_a.columns).intersection(set(df_b.columns))
    print(f"Overlapping Samples found: {overlap_samples}")
    
    # Drop overlapping samples from Collection A
    df_a_cleaned = df_a.drop(columns=list(overlap_samples))
    
    return df_a_cleaned, df_b

df_a_unique, df_b_unique = resolve_overlaps(df_a, df_b)
print(f"Collection A (Cleaned) Shape: {df_a_unique.shape}")

## 3. Missing Value Filtering

Following the "50% Rule" from Paper 1: We drop any gene (row) that contains more than 50% missing values across the samples.

In [ ]:
def apply_fifty_percent_rule(df):
    """
    Drops genes with > 50% missing values.
    """
    threshold = 0.5
    missing_mask = df.isna().sum(axis=1) / df.shape[1] > threshold
    dropped_genes = missing_mask[missing_mask].index.tolist()
    df_filtered = df.drop(index=dropped_genes)
    
    print(f"Dropped {len(dropped_genes)} genes due to >50% missing values.")
    return df_filtered

# Combine temporarily to apply filtering across the unified set, or apply individually
df_combined_raw = pd.concat([df_a_unique, df_b_unique], axis=1)
df_filtered = apply_fifty_percent_rule(df_combined_raw)
print(f"Filtered Matrix Shape: {df_filtered.shape}")

## 4. Normalization Strategy

Collection A uses Log2, while Collection B uses Scaled Min-Max. We implement the Paper 2 formula required for generative modeling:

$$x^i = \frac{x_i - \max(x_i)}{\max(x_i) - \min(x_i)} \times (High - Low) + Low$$

Where $High = 1$ and $Low = 10^{-8}$. The $Low$ value of $10^{-8}$ is explicitly used to distinguish true zeros from padding in generative models.

In [ ]:
def normalize_sample(x, high=1, low=1e-8):
    """
    Implements Scaled Min-Max normalization for a sample series.
    """
    max_val = x.max()
    min_val = x.min()
    
    # Avoid division by zero if all values are identical
    if max_val == min_val:
        return pd.Series([low] * len(x), index=x.index)
    
    normalized = ((x - max_val) / (max_val - min_val)) * (high - low) + low
    return normalized

def apply_normalization(df):
    """
    Applies normalization sample-wise (column-wise).
    """
    return df.apply(normalize_sample, axis=0)

df_normalized = apply_normalization(df_filtered)
print("Normalization complete.")

## 5. Final Output

The final step is to merge any remaining components and save the unified matrix to a CSV file.

In [ ]:
# Final shape check
print(f"Final Merged Matrix Shape: {df_normalized.shape}")

# Export to CSV
output_file = "merged_pRNA_matrix.csv"
df_normalized.to_csv(output_file)
print(f"Successfully saved to {output_file}")